In [25]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import joblib

df = pd.read_excel("cars_data.xls", na_values=["-"])
data = df.drop(["Make", "Model", "Trim"], axis=1)

changes = {"Convertible": 0, "Coupe": 1, "Hatchback": 2, "Sedan": 3, "Wegon": 4}
data["Type"] = data["Type"].map(changes)
data = data.dropna()
inv_changes = {j: i for i, j in changes.items()}
data["Type"] = data["Type"].map(inv_changes)

data.tail()

,Price,Mileage,Type,Cylinder,Liter,Doors,Cruise,Sound,Leather
799,16507.070267,16229,Sedan,6,3.0,4,1,0,0
800,16175.957604,19095,Sedan,6,3.0,4,1,1,0
801,15731.132897,20484,Sedan,6,3.0,4,1,1,0
802,15118.893228,25979,Sedan,6,3.0,4,1,1,0
803,13585.636802,35662,Sedan,6,3.0,4,1,0,0


In [26]:
X = data.drop(["Price"], axis=1)
Y = data["Price"].round(2)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2, random_state=33)

In [27]:
rf_model = RandomForestRegressor(n_estimators=140, max_depth=7, min_samples_leaf=2)
xg_model = XGBRegressor(n_estimators=110, learning_rate=0.13, max_depth=3)

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
pre = ColumnTransformer([("Cat", encoder, ["Type"])], remainder="passthrough")

model = VotingRegressor(estimators=[("rf", rf_model), ("xg", xg_model)], weights=[0.6, 0.4])

pipe = Pipeline([("preprocess", pre), ("model", model)])
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)

In [34]:
r2 = r2_score(y_test, y_pred)
print(f"finally-test-r2 : {round(r2, 4)}")

finally-test-r2 : 0.9263


In [33]:
y_pred_train = pipe.predict(x_train)
r2 = r2_score(y_train, y_pred_train)
print(f"finally-train-r2 : {round(r2, 4)}")

finally-train-r2 : 0.9462


In [32]:
joblib.dump(pipe, "ensemble.pkl")

['ensemble.pkl']